# Lab 18-05 · Serial Links: Divisors, a UART Driver and a Receiver in Verilog

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import re
import shutil
import subprocess
import tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import serialbus as sb
import fwrun

## Part 1 · Divisors

*Predict* which clock and baud-rate pairs give errors above 2% with an integer 16x divisor.

In [2]:
print(f"{'clock (MHz)':>11} {'baud':>9} {'int div':>8} {'int err %':>10} {'frac div':>9} {'frac err %':>10}")
for f in (16e6, 48e6, 14.7456e6):
    for b in (9600, 115200, 1_000_000):
        print(f"{f / 1e6:11.4f} {b:9d} {sb.baud_divisor(f, b):8.0f} {100 * sb.baud_error(f, b):10.3f} "
              f"{sb.baud_divisor(f, b, 16, 4):9.4f} {100 * sb.baud_error(f, b, 16, 4):10.3f}")
print("(a 16x divisor below 1, as for 14.7456 MHz at 1 Mbaud, is not realizable)")

clock (MHz)      baud  int div  int err %  frac div frac err %
    16.0000      9600      104      0.160  104.1875     -0.020
    16.0000    115200        9     -3.549    8.6875     -0.080
    16.0000   1000000        1      0.000    1.0000      0.000
    48.0000      9600      312      0.160  312.5000      0.000
    48.0000    115200       26      0.160   26.0625     -0.080
    48.0000   1000000        3      0.000    3.0000      0.000
    14.7456      9600       96      0.000   96.0000      0.000
    14.7456    115200        8      0.000    8.0000      0.000
    14.7456   1000000        1     -7.840    0.9375     -1.696
(a 16x divisor below 1, as for 14.7456 MHz at 1 Mbaud, is not realizable)


## Part 2 · The receiver's window

*Predict* the first and last passing ratios on the 0.0025 grid for the stock receiver (`tick == 7`) and for `tick == 6`.

In [3]:
def run_tb(rx_source):
    with tempfile.TemporaryDirectory() as td:
        subprocess.run(["iverilog", "-g2012", "-o", f"{td}/sim", str(rx_source), "hdl/uart_rx_tb.v"], check=True)
        out = subprocess.run(["vvp", "-n", f"{td}/sim"], check=True, capture_output=True, text=True).stdout
    passes = [(float(m.group(1)), m.group(2) == "pass") for m in re.finditer(r"r = ([0-9.]+) : (pass|FAIL)", out)]
    return passes, re.search(r"PASS RANGE.*", out).group(0)

if shutil.which("iverilog"):
    stock, line7 = run_tb(Path("hdl/uart_rx.v"))
    src = Path("hdl/uart_rx.v").read_text().replace("tick == 4'd7) begin", "tick == 4'd6) begin")
    tmp = Path(tempfile.mkdtemp()) / "uart_rx6.v"
    tmp.write_text(src)
    moved, line6 = run_tb(tmp)
    print("tick 7:", line7, "| theory", sb.uart_ratio_window(10, 16))
    print("tick 6:", line6, "| theory", ((9 + 7 / 16 + 1 / 16) / 10, (9 + 7 / 16) / 9))
    for data, y, lab in ((stock, 1, "tick 7 (s = 8/16)"), (moved, 0, "tick 6 (s = 7/16)")):
        r = np.array([d[0] for d in data]); ok = np.array([d[1] for d in data])
        plt.scatter(r[ok], np.full(ok.sum(), y), marker="s", label=lab + " pass")
        plt.scatter(r[~ok], np.full((~ok).sum(), y), marker="x", color="grey")
    plt.yticks([0, 1], ["tick 6", "tick 7"]); plt.xlabel("sender bit time / receiver bit time, r"); plt.legend(); plt.show()
else:
    print("iverilog not installed: see SETUP.md")

tick 7: PASS RANGE r_min 0.9575 r_max 1.0550 | theory (0.95625, 1.0555555555555556)
tick 6: PASS RANGE r_min 0.9500 r_max 1.0475 | theory (0.95, 1.0486111111111112)


## Part 3 · The UART driver

*Predict* the divisor `uart_init(115200)` computes from 25 MHz, and the resulting baud rate.

In [4]:
for opt in ("-O2", "-O0"):
    status, out, _ = fwrun.build_and_run(Path(tempfile.mkdtemp(prefix="lab1805_")), opt)
    print(opt, "exit status", status)
    print(out)

-O2 exit status 0
UART0 BAUDDIV 217
UART0 actual baud 115207
hello from the CMSDK UART
TEST PASSED



-O0 exit status 0
UART0 BAUDDIV 217
UART0 actual baud 115207
hello from the CMSDK UART
TEST PASSED



## Part 4 · I²C pull-ups

*Predict* where each speed mode's pull-up window closes on a 3.3 V bus.

In [5]:
cb = np.linspace(10e-12, 1200e-12, 500)
for name, tr, iol, cmax in (("standard", 1000e-9, 3e-3, 400e-12), ("fast", 300e-9, 3e-3, 400e-12), ("fast-mode plus", 120e-9, 20e-3, 550e-12)):
    rmin = sb.i2c_rp_min(3.3, 0.4, iol)
    line, = plt.plot(cb * 1e12, sb.i2c_rp_max(tr, cb), label=f"{name}: R_p max")
    plt.axhline(rmin, color=line.get_color(), linestyle=":")
    close = tr / (np.log(7 / 3) * rmin)
    print(f"{name:15s}: R_p min {rmin:6.0f} ohm, window closes at {close * 1e12:5.0f} pF (spec limit {cmax * 1e12:.0f} pF)")
plt.yscale("log"); plt.xlabel("bus capacitance (pF)"); plt.ylabel("pull-up (ohm)"); plt.legend(); plt.show()

standard       : R_p min    967 ohm, window closes at  1221 pF (spec limit 400 pF)
fast           : R_p min    967 ohm, window closes at   366 pF (spec limit 400 pF)
fast-mode plus : R_p min    145 ohm, window closes at   977 pF (spec limit 550 pF)


## Part 5 · SPI timing

*Predict* how much the 3 ns translators cost for a fast target (small $t_{CO}$) and for a slow one.

In [6]:
tco = np.linspace(5e-9, 40e-9, 100)
f0 = sb.spi_max_clock(2e-9, 1e-9, tco, 5e-9)
f1 = sb.spi_max_clock(2e-9, 1e-9 + 3e-9, tco, 5e-9)
plt.plot(tco * 1e9, f0 / 1e6, label="direct"); plt.plot(tco * 1e9, f1 / 1e6, label="with 3 ns translators")
plt.axhline(32, color="grey", linestyle=":", label="MCU SPI rating")
plt.xlabel("target clock-to-output (ns)"); plt.ylabel("highest SCK (MHz)"); plt.legend(); plt.show()
print("t_CO = 15 ns:", round(sb.spi_max_clock(2e-9, 1e-9, 15e-9, 5e-9) / 1e6, 1), "MHz direct,",
      round(sb.spi_max_clock(2e-9, 4e-9, 15e-9, 5e-9) / 1e6, 1), "MHz with translators (Example 18-05.4)")

t_CO = 15 ns: 20.8 MHz direct, 16.7 MHz with translators (Example 18-05.4)


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).